# Web Search quickstart

Demonstrates Azure OpenAI Responses API with the `web_search` tool using environment variables loaded via `.env`.


## Prerequisites
- Install `openai` (>=1.56.0) and `python-dotenv`.
- Populate `.env` with your Azure OpenAI settings. Supported keys (first match wins):
  - Base URL: `AZURE_OPENAI_BASE_URL` | `AZURE_OPENAI_API_BASE` | `AZURE_EXISTING_AIPROJECT_ENDPOINT`
  - Azure auth: `DefaultAzureCredential` + `get_bearer_token_provider` (run `az login` locally or use managed identity).
  - Model deployment (optional): `AZURE_OPENAI_MODEL` | `AZURE_OPENAI_DEPLOYMENT` (default `gpt-4.1`)
  - Country code (optional): `AZURE_OPENAI_COUNTRY` (for location-biased search)
  - Deep research deployment (optional): `AZURE_OPENAI_DEEP_MODEL`
- Using web search can incur costs and sends data to Bing for grounding.


In [ ]:
# Optional setup: install dependencies if the kernel is missing them.
# If needed, install dependencies (uncomment to run)
# %pip install -U openai python-dotenv


In [10]:
# Load environment variables and normalize the Azure OpenAI base URL.
import os
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from dotenv import load_dotenv

load_dotenv(override=True)

base_url = (
    os.getenv("AZURE_OPENAI_BASE_URL")
    or os.getenv("AZURE_OPENAI_ENDPOINT")
    or os.getenv("AZURE_EXISTING_AIPROJECT_ENDPOINT")
)
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)
model = os.getenv("AZURE_OPENAI_MODEL") or os.getenv("AZURE_OPENAI_DEPLOYMENT") or "gpt-5.2"
print(model)
country = os.getenv("AZURE_OPENAI_COUNTRY")
deep_model = os.getenv("AZURE_OPENAI_DEEP_MODEL")
# Optionally use a deep research model if configured in the environment.
print(deep_model)

if not base_url:
    raise ValueError("Set base URL in .env (see prerequisites).")

base_url = base_url.rstrip("/")
if not base_url.endswith("/openai/v1"):
    base_url = f"{base_url}/openai/v1"


gpt-5-mini
None


In [11]:
# Instantiate the OpenAI client for the Responses API.
from openai import OpenAI

client = OpenAI(api_key=token_provider, base_url=base_url)
print(f"Client ready against {base_url}")


Client ready against https://jacwang-foundryonly.cognitiveservices.azure.com/openai/v1


In [12]:
# Helper utilities for invoking web search and printing structured output.
# This keeps the demo calls consistent and easy to compare.
import json
from typing import Optional, List

def run_search(prompt: str, tools: List[dict], model_override: Optional[str] = None):
    # Submit the prompt with the selected tools and capture the response.
    response = client.responses.create(
        model=model_override or model,
        tools=tools,
        input=prompt,
        # reasoning_effort="high",
        tool_choice="required" #Optional if you want to enforce tool calling
    )

    # Print the main response text for quick inspection.
    print(response.output_text)

    # Display annotations (citations) if present
    messages = [item for item in response.output or [] if item.type == "message"]
    for message in messages:
        for content in message.content or []:
            if hasattr(content, 'annotations') and content.annotations:
                print("\n" + "=" * 80)
                print("Citations:")
                print("=" * 80)
                for i, annotation in enumerate(content.annotations, 1):
                    # Extract the cited text using start_index and end_index
                    if hasattr(annotation, 'start_index') and hasattr(annotation, 'end_index'):
                        cited_text = content.text[annotation.start_index:annotation.end_index]
                        print(f"\n[{i}] {cited_text}")
                    else:
                        print(f"\n[{i}] Citation {i}")
                    
                    # Display URL and title if available
                    if hasattr(annotation, 'url') and annotation.url:
                        print(f"    URL: {annotation.url}")
                    if hasattr(annotation, 'title') and annotation.title:
                        print(f"    Title: {annotation.title}")
                print("=" * 80)

    # Extract and display explicit tool calls for debugging.
    actions = [item for item in response.output or [] if item.type == "web_search_call"]
    if actions:
        print("\n" + "=" * 80)
        print("Tool calls:")
        print("=" * 80)
        for i, action in enumerate(actions, 1):
            try:
                print(f"\nCall {i}:")
                print(json.dumps(action.model_dump(), indent=2, ensure_ascii=False, sort_keys=False))
            except Exception:
                print(action)
        print("=" * 80)

    return response


### Quick lookup (single search)
Uses the fastest path: the model forwards the prompt directly to web search without multi-step planning.


In [14]:
# Repeat the quick lookup to compare response variability.
run_search(
    "what is the latest with JPMorgan Chase?",
    tools=[{"type": "web_search"}]
    ,
)

I can get you a quick update — do you mean recent news, the stock price, latest earnings, or regulatory/legal developments? I pulled the latest headlines; short summary below and I can dive deeper into any item.

- CEO Jamie Dimon publicly attacked the crypto industry and criticized Coinbase’s CEO as part of an opposition to the CLARITY Act — those comments were reported June 2, 2026. ([finance.yahoo.com](https://finance.yahoo.com/markets/crypto/articles/jamie-dimon-calls-coinbase-ceo-171657206.html))  
- JPMorgan reported first-quarter 2026 results on April 14, 2026: net income of $16.5 billion (about $5.94 per share); the company reported $4.9 trillion in assets as of March 31, 2026. ([jpmorgan.com](https://www.jpmorgan.com/about-us/corporate-news/2026))  
- Corporate updates: Chase is expanding Sapphire Reserve airport-lounge access and announced new community/philanthropic investments (e.g., Detroit) in recent press items (May 2026). ([media.chase.com](https://media.chase.com/news)

Response(id='resp_02fbcbc624ac7cb2006a205c316dd08193a6df248d17aae08b', created_at=1780505649.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5-mini', object='response', output=[ResponseReasoningItem(id='rs_02fbcbc624ac7cb2006a205c31e81c8193a9fa0e11d93e3ed6', summary=[], type='reasoning', content=None, encrypted_content=None, status=None), ResponseFunctionWebSearch(id='ws_02fbcbc624ac7cb2006a205c349ec881939a8e05a3f8f29645', action=ActionSearch(query='JPMorgan Chase latest news June 2026', type='search', sources=None, queries=['JPMorgan Chase latest news June 2026', 'JPMorgan Chase earnings May 2026 Q2 2026', 'JPMorgan CEO Jamie Dimon June 2026 news', 'JPMorgan Chase legal issues 2026 settlement fine June 2026']), status='completed', type='web_search_call'), ResponseReasoningItem(id='rs_02fbcbc624ac7cb2006a205c3704b481939dcf0124f9dda1f1', summary=[], type='reasoning', content=None, encrypted_content=None, status=None), ResponseOutputMessage(id='msg_02fb

In [15]:
# Repeat the quick lookup to compare response variability.
run_search(
    "\nYou are a factual assistant.\n\n- Use the Bing grounding tool to search the web to answer the question\n- Cite all web sources you use to answer the question\n- If no relevant data is found, state exactly: \"I cannot find any relevant search results.\"\n- Answer directly and factually; do not include greetings or introductory text.\n\n",
    tools=[{"type": "web_search"}]
    ,
)

AzureCliCredential.get_token_info failed: Failed to invoke the Azure CLI


CredentialUnavailableError: Failed to invoke the Azure CLI

In [10]:
# Demonstrate location-biased search results via user_location.
location_tools = [
    {
        "type": "web_search",
        "user_location": {"type": "approximate", "country": country or "US"},
    }
]

run_search(
    "Share a positive news story from the web today.",
    tools=location_tools,
)


Here is a positive news story from today:

An Indian teacher named Rouble Nagi has won the $1 million Global Teacher Prize from GEMS Education for her incredible work of establishing more than 800 learning centers across India. These centers offer education to children in slums and villages who have never attended school, providing safe and inspiring spaces despite the challenges of poverty, child labor, and lack of infrastructure. Her efforts have created many educational opportunities and hope for disadvantaged children in India.

You can read more about her inspiring story here: [Good News Network - Teacher wins $1M prize for turning India's slums into hundreds of open-air classrooms](https://www.sunnyskyz.com/good-news)

Citations:

[1] [Good News Network - Teacher wins $1M prize for turning India's slums into hundreds of open-air classrooms](https://www.sunnyskyz.com/good-news)
    URL: https://www.sunnyskyz.com/good-news
    Title: Good News Stories - Sunny Skyz

Tool calls:

Cal

Response(id='resp_0b40a05e44a37f9f0069ae4bee7968819582cba619acdb8961', created_at=1773030382.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4.1-mini', object='response', output=[ResponseFunctionWebSearch(id='ws_0b40a05e44a37f9f0069ae4beebc3481958311170fee8017df', action=ActionSearch(query='positive news story March 9, 2026', type='search', sources=None, queries=['positive news story March 9, 2026']), status='searching', type='web_search_call'), ResponseOutputMessage(id='msg_0b40a05e44a37f9f0069ae4bef82988195aee188fdbfc44e05', content=[ResponseOutputText(annotations=[AnnotationURLCitation(end_index=733, start_index=589, title='Good News Stories - Sunny Skyz', type='url_citation', url='https://www.sunnyskyz.com/good-news')], text="Here is a positive news story from today:\n\nAn Indian teacher named Rouble Nagi has won the $1 million Global Teacher Prize from GEMS Education for her incredible work of establishing more than 800 learning centers across In

In [ ]:
# Placeholder cell for additional experiments or notes.
